This notebook can be used to explore the memory usage on a MCU if micropython app prints the meminfo to the serial port.

Logging can be done using the following code snippet:
```
import micropython
micropython.mem_info(1)
```
The output of this command can be captured by the notebook and displayed as a memory map.

For more advanced logging, the `MemInfoLogger` class can be used as a decorator or called independently.
See that class in `setup_meminfologger.ipynb` for details.


In [1]:
!mpremote rtc --set

In [2]:
%run setup_meminfologger.ipynb

MemInfoLogger set up.


In [3]:
# %%micropython

@MemInfoLogger()  # Will use "do_work" as label automatically
def do_work():
    block = bytearray(500 + random.randint(0, 1000))
    return block


chain = []
for n in range(100):
    do_work()
    chain.append(do_work())
    time.sleep_ms(10)

*** Memory info do_work after ***
time:1767307979856502000
stack: 1164 out of 7936
GC: total: 205568, used: 7488, free: 198080
 No. of 1-blocks: 71, 2-blocks: 22, max blk sz: 79, max free sz: 12297
***********************
*** Memory info do_work after ***
time:1767307979864964000
stack: 1164 out of 7936
GC: total: 205568, used: 8608, free: 196960
 No. of 1-blocks: 84, 2-blocks: 24, max blk sz: 79, max free sz: 12247
***********************
*** Memory info do_work after ***
time:1767307979884006000
stack: 1164 out of 7936
GC: total: 205568, used: 7728, free: 197840
 No. of 1-blocks: 69, 2-blocks: 21, max blk sz: 64, max free sz: 12193
***********************
*** Memory info do_work after ***
time:1767307979892231000
stack: 1164 out of 7936
GC: total: 205568, used: 9136, free: 196432
 No. of 1-blocks: 82, 2-blocks: 23, max blk sz: 68, max free sz: 12193
***********************
*** Memory info do_work after ***
time:1767307979909880000
stack: 1164 out of 7936
GC: total: 205568, used: 1017

In [4]:
# use the last output, convert it to a list and store in log_text
output = _
log_text = list(output)
%store log_text 

Stored 'log_text' (list)


In [5]:
del log_text, output

In [1]:
# retreieve log_text from storage
%store -r log_text

In [2]:
from micropython_magic.memoryinfo import MemoryInfoList, MemoryInfo

In [5]:
mi_list = MemoryInfoList(show_free=False, columns=2)
mi_list.parse_log(log_text)

do_work after
Stack used:  0x048c of Total: 0x1f00  pct free: 85.3%
Memory used: 0x001_b490 of Total: 0x003_2300  free: 0x001_6e70 pct free: 45.6%
1-Blocks:         179  2-Blocks:         23
Max Block size:    94  Max Free size:  5379


In [ ]:
mi_list.columns = 1
mi_list

do_work after
Stack used:  0x048c of Total: 0x1f00  pct free: 85.3%
Memory used: 0x001_b490 of Total: 0x003_2300  free: 0x001_6e70 pct free: 45.6%
1-Blocks:         179  2-Blocks:         23
Max Block size:    94  Max Free size:  5379


In [6]:
mi_list.columns = 4
handle = display(mi_list, display_id=True)

do_work after
Stack used:  0x048c of Total: 0x1f00  pct free: 85.3%
Memory used: 0x001_b490 of Total: 0x003_2300  free: 0x001_6e70 pct free: 45.6%
1-Blocks:         179  2-Blocks:         23
Max Block size:    94  Max Free size:  5379


In [7]:
from IPython.display import display, update_display

mi_list.columns = 2
mi_list.columns
# did = display(mil, display_id=True)
# update_display(mil, display_id=did.display_id)

2

In [8]:
display(mi_list)

do_work after
Stack used:  0x048c of Total: 0x1f00  pct free: 85.3%
Memory used: 0x001_b490 of Total: 0x003_2300  free: 0x001_6e70 pct free: 45.6%
1-Blocks:         179  2-Blocks:         23
Max Block size:    94  Max Free size:  5379


In [9]:
display("", display_id="here")
for mm in mi_list:
    update_display(mm, display_id="here")

do_work after
Stack used:  0x048c of Total: 0x1f00  pct free: 85.3%
Memory used: 0x001_b490 of Total: 0x003_2300  free: 0x001_6e70 pct free: 45.6%
1-Blocks:         179  2-Blocks:         23
Max Block size:    94  Max Free size:  5379


In [10]:
# difference between the first and last memory info
if mi_list and len(mi_list) > 2:
    diff = mi_list[-1] - mi_list[0]
    diff